# BDLegalRAG — Stage 5: Hybrid Retrieval (RQ2)

Stage 3-4 থেকে চূড়ান্ত হয়েছে: **Section-aware chunking + BGE-M3 dense embedding**। এই Stage-এ তার সাথে **BM25 (sparse/lexical) retrieval** যোগ করে **Reciprocal Rank Fusion (RRF)** দিয়ে combine করা হবে।

**তুলনা করা হবে তিনটা configuration:**
1. BM25-only (sparse)
2. BGE-M3-only (dense — Stage 3/4-এর baseline)
3. **Hybrid (BM25 + BGE-M3, RRF fusion)** — মূল contribution

**লক্ষ্য:** Stage 3-এর failure analysis-এ পাওয়া "boilerplate/generic clause confusion" pattern (qa_0099, qa_0129, qa_0144, qa_0158) — যেখানে dense embedding একই রকম ভাষার আইনে বিভ্রান্ত হয়েছিল — সেটা BM25-এর exact keyword matching দিয়ে কমে কিনা যাচাই করা।

**নোট:** BGE-M3-এর নিজস্ব native sparse mode আছে, কিন্তু সেটা ব্যবহার করতে `FlagEmbedding` library এবং custom sparse-matrix retrieval infrastructure লাগে (sentence-transformers দিয়ে পাওয়া যায় না)। Reliability আর simplicity-র জন্য standard, well-established **rank_bm25** ব্যবহার করা হচ্ছে — এটাই literature-এর প্রায় সব hybrid RAG paper-এ ব্যবহৃত পদ্ধতি।


## ধাপ ০: Setup

In [ ]:
import sys, os, json, re, pickle

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/BDLegalRAG"
else:
    BASE_DIR = "/kaggle/working/BDLegalRAG"

PROCESSED_DIR = os.path.join(BASE_DIR, "processed_data")
QA_DIR = os.path.join(BASE_DIR, "qa_data")
EMBED_DIR = os.path.join(BASE_DIR, "embedding_benchmark")   # Stage 4-এর cache এখান থেকে reuse হবে
HYBRID_DIR = os.path.join(BASE_DIR, "hybrid_retrieval")
os.makedirs(HYBRID_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/BDLegalRAG


In [ ]:
!pip install sentence-transformers faiss-cpu rank_bm25 pandas --quiet
import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

print("Libraries ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.2 MB/s eta 0:00:00
Libraries ready.


In [ ]:
with open(os.path.join(PROCESSED_DIR, "bdlegalrag_corpus_v1.json"), encoding="utf-8") as f:
    corpus = json.load(f)

gold_df = pd.read_csv(os.path.join(QA_DIR, "gold_qa_set_v1.csv"))

print("Total acts:", len(corpus["acts"]))
print("Total gold QA pairs:", len(gold_df))


Total acts: 49
Total gold QA pairs: 180


## ধাপ ১: Section-Aware Chunks + Digit-Normalization Fix (Stage 3/4 থেকে reuse)

In [ ]:
all_sections = []
for act in corpus["acts"]:
    for sec in act["sections"]:
        all_sections.append({
            "act_title": act["act_title"],
            "act_no": act.get("act_no"),
            "section_number": sec.get("section_number"),
            "section_text": sec["section_text"],
        })

chunks = [{**s, "chunk_text": s["section_text"]} for s in all_sections]
print(f"Total chunks: {len(chunks)}")

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

def normalize_section_number(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    return str(val).strip().translate(BN_TO_EN)

def is_match(chunk, gold_row):
    same_act = str(chunk["act_title"]).strip() == str(gold_row["act_title"]).strip()
    same_section = normalize_section_number(chunk["section_number"]) == normalize_section_number(gold_row["section_number"])
    return same_act and same_section

print("Matching logic ready.")


Total chunks: 1575
Matching logic ready.


## ধাপ ২: BM25 Index বানান (Sparse Retrieval)

বাংলা-সচেতন সাধারণ tokenization (Bangla Unicode block + Latin + সংখ্যা ধরে শব্দ আলাদা করা) ব্যবহার করা হচ্ছে।


In [ ]:
def tokenize(text):
    return re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", text.lower())


tokenized_chunks = [tokenize(c["chunk_text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index built over {len(tokenized_chunks)} chunks.")


BM25 index built over 1575 chunks.


## ধাপ ৩: BGE-M3 Dense Index লোড করুন (Stage 4-এর Cache Reuse)

Stage 4-এ ইতিমধ্যে BGE-M3 দিয়ে সব chunk embed করা আছে — সেই cache থেকেই সরাসরি নেওয়া হচ্ছে, আবার embed করার দরকার নেই।


In [ ]:
STAGE4_CACHE_PATH = os.path.join(EMBED_DIR, "embedding_benchmark_cache.pkl")

if os.path.exists(STAGE4_CACHE_PATH):
    print("Stage 4-এর cache পাওয়া গেছে, reuse করা হচ্ছে...")
    with open(STAGE4_CACHE_PATH, "rb") as f:
        stage4_cache = pickle.load(f)
    dense_embeddings = np.asarray(stage4_cache["bge_m3"], dtype="float32")
    dense_model = SentenceTransformer("BAAI/bge-m3")  # query embed করতে লাগবে
else:
    print("Stage 4-এর cache পাওয়া যায়নি — নতুন করে BGE-M3 দিয়ে embed করা হচ্ছে (সময় লাগবে)...")
    dense_model = SentenceTransformer("BAAI/bge-m3")
    texts = [c["chunk_text"] for c in chunks]
    dense_embeddings = dense_model.encode(
        texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True
    )
    dense_embeddings = np.asarray(dense_embeddings, dtype="float32")

dense_index = faiss.IndexFlatIP(dense_embeddings.shape[1])
dense_index.add(dense_embeddings)
print(f"Dense index ready: {dense_index.ntotal} vectors")


Stage 4-এর cache পাওয়া গেছে, reuse করা হচ্ছে...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dense index ready: 1575 vectors


## ধাপ ৪: BM25, Dense, এবং RRF Fusion দিয়ে Retrieval Function

**Reciprocal Rank Fusion (RRF)** — দুটো ranking-কে rank-ভিত্তিক score দিয়ে মেলানো হয় (raw score না, কারণ BM25 আর cosine similarity-র scale সম্পূর্ণ আলাদা, সরাসরি যোগ করা যায় না):

`RRF_score(chunk) = 1/(k + rank_bm25) + 1/(k + rank_dense)` — এখানে `k=60`, literature-এ standard default।


In [ ]:
RRF_K = 60

def retrieve_bm25(query, top_n=50):
    scores = bm25.get_scores(tokenize(query))
    ranked_idx = np.argsort(scores)[::-1][:top_n]
    return list(ranked_idx)


def retrieve_dense(query, top_n=50):
    q_emb = dense_model.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    _, idx = dense_index.search(q_emb, top_n)
    return list(idx[0])


def retrieve_hybrid_rrf(query, top_n=50, pool_size=50):
    bm25_ranked = retrieve_bm25(query, top_n=pool_size)
    dense_ranked = retrieve_dense(query, top_n=pool_size)

    rrf_scores = {}
    for rank, idx in enumerate(bm25_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)
    for rank, idx in enumerate(dense_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked[:top_n]]

print("Retrieval functions ready: BM25-only, Dense-only, Hybrid-RRF.")


Retrieval functions ready: BM25-only, Dense-only, Hybrid-RRF.


## ধাপ ৫: তিনটা Configuration Evaluate করুন

In [ ]:
def evaluate_config(retrieve_fn, k_values=(5, 10, 20)):
    max_k = max(k_values)
    results = {k: {"recall_hits": 0, "precision_sum": 0.0} for k in k_values}
    reciprocal_ranks = []

    for _, gold_row in gold_df.reset_index(drop=True).iterrows():
        retrieved_idx = retrieve_fn(gold_row["question"], top_n=max_k)
        retrieved_chunks = [chunks[i] for i in retrieved_idx]

        rr = 0.0
        for rank, ch in enumerate(retrieved_chunks, start=1):
            if is_match(ch, gold_row):
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

        for k in k_values:
            top_k = retrieved_chunks[:k]
            hit = any(is_match(ch, gold_row) for ch in top_k)
            results[k]["recall_hits"] += int(hit)
            results[k]["precision_sum"] += (1 if hit else 0) / k

    n = len(gold_df)
    summary = {"MRR": sum(reciprocal_ranks) / n}
    for k in k_values:
        summary[f"Recall@{k}"] = results[k]["recall_hits"] / n
        summary[f"Precision@{k}"] = results[k]["precision_sum"] / n
    return summary


configs = {
    "bm25_only": retrieve_bm25,
    "dense_only_bge_m3": retrieve_dense,
    "hybrid_rrf": retrieve_hybrid_rrf,
}

all_results = []
for name, fn in configs.items():
    print(f"Evaluating: {name}")
    summary = evaluate_config(fn)
    summary["config"] = name
    all_results.append(summary)
    print(summary, "\n")


Evaluating: bm25_only
{'MRR': 0.44078752266252264, 'Recall@5': 0.5722222222222222, 'Precision@5': 0.11444444444444421, 'Recall@10': 0.6666666666666666, 'Precision@10': 0.06666666666666651, 'Recall@20': 0.7277777777777777, 'Precision@20': 0.036388888888888804, 'config': 'bm25_only'} 

Evaluating: dense_only_bge_m3
{'MRR': 0.7117894417101073, 'Recall@5': 0.8222222222222222, 'Precision@5': 0.16444444444444403, 'Recall@10': 0.8777777777777778, 'Precision@10': 0.08777777777777755, 'Recall@20': 0.9277777777777778, 'Precision@20': 0.0463888888888888, 'config': 'dense_only_bge_m3'} 

Evaluating: hybrid_rrf
{'MRR': 0.5984314268415146, 'Recall@5': 0.7611111111111111, 'Precision@5': 0.15222222222222187, 'Recall@10': 0.8333333333333334, 'Precision@10': 0.08333333333333312, 'Recall@20': 0.9111111111111111, 'Precision@20': 0.04555555555555545, 'config': 'hybrid_rrf'} 



## ধাপ ৬: ফলাফল তুলনা টেবিল — RQ2-এর মূল Result

In [ ]:
results_df = pd.DataFrame(all_results).set_index("config")
results_df = results_df[["MRR", "Recall@5", "Precision@5", "Recall@10", "Precision@10", "Recall@20", "Precision@20"]]
results_df = results_df.round(4)

print(results_df.to_string())

results_path = os.path.join(HYBRID_DIR, "hybrid_retrieval_results.csv")
results_df.to_csv(results_path)
print(f"\nSaved to: {results_path}")


                      MRR  Recall@5  Precision@5  Recall@10  Precision@10  Recall@20  Precision@20
config                                                                                            
bm25_only          0.4408    0.5722       0.1144     0.6667        0.0667     0.7278        0.0364
dense_only_bge_m3  0.7118    0.8222       0.1644     0.8778        0.0878     0.9278        0.0464
hybrid_rrf         0.5984    0.7611       0.1522     0.8333        0.0833     0.9111        0.0456

Saved to: /content/drive/MyDrive/BDLegalRAG/hybrid_retrieval/hybrid_retrieval_results.csv


## [EXPERIMENT] Weighted RRF — দুর্বল BM25-কে কম ওজন দিয়ে

যেহেতু BM25 (MRR 0.44) BGE-M3 (MRR 0.71)-এর চেয়ে যথেষ্ট দুর্বল, standard equal-weight RRF দুর্বল signal-কে সমান গুরুত্ব দিয়ে সামগ্রিক ফলাফল খারাপ করে দিচ্ছে। এখানে dense-কে বেশি ওজন দিয়ে কয়েকটা ratio try করে দেখা হচ্ছে কোনটায় dense-only-কে ছাড়িয়ে যাওয়া যায় কিনা।


In [ ]:
def retrieve_hybrid_weighted_rrf(query, dense_weight, bm25_weight, top_n=50, pool_size=50):
    bm25_ranked = retrieve_bm25(query, top_n=pool_size)
    dense_ranked = retrieve_dense(query, top_n=pool_size)

    rrf_scores = {}
    for rank, idx in enumerate(bm25_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + bm25_weight / (RRF_K + rank + 1)
    for rank, idx in enumerate(dense_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + dense_weight / (RRF_K + rank + 1)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked[:top_n]]


def evaluate_weighted(dense_weight, bm25_weight, k_values=(5, 10, 20)):
    fn = lambda q, top_n: retrieve_hybrid_weighted_rrf(q, dense_weight, bm25_weight, top_n=top_n)
    return evaluate_config(fn, k_values=k_values)


weight_configs = [
    (1.0, 1.0),   # আগের equal-weight, baseline হিসেবে
    (2.0, 1.0),
    (3.0, 1.0),
    (5.0, 1.0),
    (10.0, 1.0),
]

weighted_results = []
for dw, bw in weight_configs:
    summary = evaluate_weighted(dw, bw)
    summary["config"] = f"dense_w={dw}_bm25_w={bw}"
    weighted_results.append(summary)
    print(summary["config"], "->", {k: round(v, 4) for k, v in summary.items() if k != "config"})

weighted_df = pd.DataFrame(weighted_results).set_index("config")
weighted_df = weighted_df[["MRR", "Recall@5", "Precision@5", "Recall@10", "Precision@10", "Recall@20", "Precision@20"]]
print()
print(weighted_df.round(4).to_string())

weighted_path = os.path.join(HYBRID_DIR, "weighted_rrf_sweep_results.csv")
weighted_df.to_csv(weighted_path)
print(f"\nSaved to: {weighted_path}")
print(f"\nতুলনার জন্য মনে রাখুন — dense_only_bge_m3 একাই দিয়েছিল MRR ~0.71, Recall@5 ~0.82")


dense_w=1.0_bm25_w=1.0 -> {'MRR': 0.5984, 'Recall@5': 0.7611, 'Precision@5': 0.1522, 'Recall@10': 0.8333, 'Precision@10': 0.0833, 'Recall@20': 0.9111, 'Precision@20': 0.0456}
dense_w=2.0_bm25_w=1.0 -> {'MRR': 0.6663, 'Recall@5': 0.7944, 'Precision@5': 0.1589, 'Recall@10': 0.8444, 'Precision@10': 0.0844, 'Recall@20': 0.9222, 'Precision@20': 0.0461}
dense_w=3.0_bm25_w=1.0 -> {'MRR': 0.6754, 'Recall@5': 0.7944, 'Precision@5': 0.1589, 'Recall@10': 0.8722, 'Precision@10': 0.0872, 'Recall@20': 0.9222, 'Precision@20': 0.0461}
dense_w=5.0_bm25_w=1.0 -> {'MRR': 0.6772, 'Recall@5': 0.8167, 'Precision@5': 0.1633, 'Recall@10': 0.8778, 'Precision@10': 0.0878, 'Recall@20': 0.9278, 'Precision@20': 0.0464}
dense_w=10.0_bm25_w=1.0 -> {'MRR': 0.6897, 'Recall@5': 0.8167, 'Precision@5': 0.1633, 'Recall@10': 0.8722, 'Precision@10': 0.0872, 'Recall@20': 0.9278, 'Precision@20': 0.0464}

                            MRR  Recall@5  Precision@5  Recall@10  Precision@10  Recall@20  Precision@20
config            

## ধাপ ৭: Targeted Check — Boilerplate-Confusion Cases কি ঠিক হলো?

Stage 3-এ যে ৪টা প্রশ্ন (qa_0099, qa_0129, qa_0144, qa_0158) generic legal ভাষার কারণে ভুল Act retrieve করেছিল, সেগুলো Hybrid-RRF-এ ঠিক হলো কিনা নির্দিষ্টভাবে চেক করা হচ্ছে।


In [ ]:
BOILERPLATE_CASES = ["qa_0099", "qa_0129", "qa_0144", "qa_0158"]

check_rows = gold_df[gold_df["qa_id"].isin(BOILERPLATE_CASES)] if "qa_id" in gold_df.columns else pd.DataFrame()

if len(check_rows) == 0:
    print("gold_df-এ 'qa_id' কলাম নেই বা এই ID গুলো পাওয়া যায়নি — gold_qa_set_v1.csv-এ qa_id কলাম আছে কিনা চেক করুন।")
else:
    for _, row in check_rows.iterrows():
        retrieved_idx = retrieve_hybrid_rrf(row["question"], top_n=5)
        retrieved_chunks = [chunks[i] for i in retrieved_idx]
        hit = any(is_match(ch, row) for ch in retrieved_chunks)
        status = "✓ এখন ঠিক (top-5-এ পাওয়া গেছে)" if hit else "✗ এখনো ভুল"
        print(f"{row['qa_id']}: {status}")
        print(f"  Gold: {row['act_title']} - {row['section_number']}")
        print(f"  Top-1 retrieved: {retrieved_chunks[0]['act_title']} - {retrieved_chunks[0]['section_number']}")
        print()


qa_0099: ✗ এখনো ভুল
  Gold: পারিবারিক সহিংসতা (প্রতিরোধ ও সুরক্ষা) আইন, ২০১০ - 32
  Top-1 retrieved: পারিবারিক আদালত আইন, ২০২৩ - 23

qa_0129: ✗ এখনো ভুল
  Gold: বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭ - 25
  Top-1 retrieved: বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭ - 13

qa_0144: ✗ এখনো ভুল
  Gold: স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭ - 46
  Top-1 retrieved: স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭ - 40

qa_0158: ✗ এখনো ভুল
  Gold: ভূমি সংস্কার বোর্ড আইন, ১৯৮৯ - 7
  Top-1 retrieved: পণ্য উৎপাদনশীল রাষ্ট্রায়ত্ত শিল্প প্রতিষ্ঠান শ্রমিক (চাকরির শর্তাবলী) আইন, ২০১৮ - 5



## [EXPERIMENT] Act-Name Context Enrichment

BM25 fusion কাজ করেনি, কারণ boilerplate/generic clause একাধিক Act-এ প্রায় হুবহু ভাষায় লেখা — dense embedding-ও raw section text দিয়ে এগুলো আলাদা করতে পারছে না। এখানে প্রতিটা chunk embed করার আগে **Act-এর নাম prepend** করে দেওয়া হচ্ছে, যাতে "এই clause-টা কোন Act-এর" — এই তথ্যও embedding-এ থাকে।

```
আগে:  chunk_text = section_text
এখন:  chunk_text = "{act_title} — {section_text}"
```

**নোট:** এটা নতুন embedding তৈরি করবে (আলাদা cache ফাইলে), Stage 3/4-এর আগের cache পরিবর্তন করা হচ্ছে না — যাতে দুটো তুলনা করা যায়।


In [ ]:
CONTEXT_ENRICHED_CACHE_PATH = os.path.join(HYBRID_DIR, "context_enriched_embeddings_cache.pkl")

context_enriched_texts = [f"{c['act_title']} — {c['chunk_text']}" for c in chunks]

if os.path.exists(CONTEXT_ENRICHED_CACHE_PATH):
    print("Context-enriched cache পাওয়া গেছে, আবার embed করা হচ্ছে না...")
    with open(CONTEXT_ENRICHED_CACHE_PATH, "rb") as f:
        context_embeddings = pickle.load(f)
else:
    print("নতুন করে context-enriched text embed করা হচ্ছে (BGE-M3, সময় লাগবে)...")
    context_embeddings = dense_model.encode(
        context_enriched_texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True
    )
    context_embeddings = np.asarray(context_embeddings, dtype="float32")
    with open(CONTEXT_ENRICHED_CACHE_PATH, "wb") as f:
        pickle.dump(context_embeddings, f)
    print("Cache saved.")

context_index = faiss.IndexFlatIP(context_embeddings.shape[1])
context_index.add(context_embeddings)
print(f"Context-enriched index ready: {context_index.ntotal} vectors")


def retrieve_context_enriched(query, top_n=50):
    q_emb = dense_model.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    _, idx = context_index.search(q_emb, top_n)
    return list(idx[0])


Context-enriched cache পাওয়া গেছে, আবার embed করা হচ্ছে না...
Context-enriched index ready: 1575 vectors


### ফলাফল তুলনা — Context-Enriched vs আগের Dense-only Baseline

In [ ]:
summary_context = evaluate_config(retrieve_context_enriched)
summary_context["config"] = "dense_context_enriched"
print(summary_context)

# আগের dense_only_bge_m3 ফলাফলের সাথে পাশাপাশি দেখানো
comparison_df = pd.DataFrame([
    {**[r for r in all_results if r["config"] == "dense_only_bge_m3"][0]},
    summary_context,
]).set_index("config")
comparison_df = comparison_df[["MRR", "Recall@5", "Precision@5", "Recall@10", "Precision@10", "Recall@20", "Precision@20"]]
print()
print(comparison_df.round(4).to_string())

comparison_path = os.path.join(HYBRID_DIR, "context_enrichment_comparison.csv")
comparison_df.to_csv(comparison_path)
print(f"\nSaved to: {comparison_path}")


{'MRR': 0.8249734958835837, 'Recall@5': 0.9444444444444444, 'Precision@5': 0.18888888888888858, 'Recall@10': 0.9611111111111111, 'Precision@10': 0.09611111111111098, 'Recall@20': 0.9888888888888889, 'Precision@20': 0.049444444444444395, 'config': 'dense_context_enriched'}

                           MRR  Recall@5  Precision@5  Recall@10  Precision@10  Recall@20  Precision@20
config                                                                                                 
dense_only_bge_m3       0.7118    0.8222       0.1644     0.8778        0.0878     0.9278        0.0464
dense_context_enriched  0.8250    0.9444       0.1889     0.9611        0.0961     0.9889        0.0494

Saved to: /content/drive/MyDrive/BDLegalRAG/hybrid_retrieval/context_enrichment_comparison.csv


### Boilerplate Cases-এ Targeted Check

In [ ]:
for qa_id in BOILERPLATE_CASES:
    row = gold_df[gold_df["qa_id"] == qa_id]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    retrieved_idx = retrieve_context_enriched(row["question"], top_n=5)
    retrieved_chunks = [chunks[i] for i in retrieved_idx]
    hit = any(is_match(ch, row) for ch in retrieved_chunks)
    status = "✓ এখন ঠিক (top-5-এ পাওয়া গেছে)" if hit else "✗ এখনো ভুল"
    print(f"{qa_id}: {status}")
    print(f"  Gold: {row['act_title']} - {row['section_number']}")
    print(f"  Top-1 retrieved: {retrieved_chunks[0]['act_title']} - {retrieved_chunks[0]['section_number']}")
    print()


qa_0099: ✓ এখন ঠিক (top-5-এ পাওয়া গেছে)
  Gold: পারিবারিক সহিংসতা (প্রতিরোধ ও সুরক্ষা) আইন, ২০১০ - 32
  Top-1 retrieved: বাল্যবিবাহ নিরোধ আইন, ২০১৭ - 6

qa_0129: ✓ এখন ঠিক (top-5-এ পাওয়া গেছে)
  Gold: বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭ - 25
  Top-1 retrieved: বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭ - 12

qa_0144: ✓ এখন ঠিক (top-5-এ পাওয়া গেছে)
  Gold: স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭ - 46
  Top-1 retrieved: স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭ - 46

qa_0158: ✗ এখনো ভুল
  Gold: ভূমি সংস্কার বোর্ড আইন, ১৯৮৯ - 7
  Top-1 retrieved: অর্পিত সম্পত্তি প্রত্যর্পণ আইন, ২০০১ - 28ক



In [ ]:
!pip install statsmodels --quiet
from statsmodels.stats.contingency_tables import mcnemar

def get_hits_for_fn(retrieve_fn, k=5):
    hits = []
    for _, row in gold_df.reset_index(drop=True).iterrows():
        retrieved_idx = retrieve_fn(row["question"], top_n=k)
        retrieved_chunks = [chunks[i] for i in retrieved_idx]
        hits.append(any(is_match(ch, row) for ch in retrieved_chunks))
    return np.array(hits)

hits_baseline = get_hits_for_fn(retrieve_dense, k=5)
hits_context = get_hits_for_fn(retrieve_context_enriched, k=5)

both_right = int(np.sum(hits_baseline & hits_context))
only_baseline = int(np.sum(hits_baseline & ~hits_context))
only_context = int(np.sum(~hits_baseline & hits_context))
both_wrong = int(np.sum(~hits_baseline & ~hits_context))

table = [[both_right, only_baseline], [only_context, both_wrong]]
result = mcnemar(table, exact=True)

print(f"শুধু baseline সঠিক: {only_baseline}, শুধু context-enriched সঠিক: {only_context}")
print(f"দুটোই সঠিক: {both_right}, দুটোই ভুল: {both_wrong}")
print(f"p-value: {result.pvalue:.6f}")
print("Statistically significant improvement (p < 0.05)" if result.pvalue < 0.05 else "significant না")

শুধু baseline সঠিক: 5, শুধু context-enriched সঠিক: 27
দুটোই সঠিক: 143, দুটোই ভুল: 5
p-value: 0.000113
Statistically significant improvement (p < 0.05)


FINAL


## পরের ধাপ

যদি Hybrid-RRF সব metric-এ (বিশেষত MRR এবং Precision@5) BGE-M3-only-কে ছাড়িয়ে যায়, এটাই আপনার **RQ2-এর মূল finding** এবং চূড়ান্ত retrieval configuration।

**Stage 6 (Cross-Encoder Reranking)**-এ, এই Hybrid-RRF থেকে top-20 candidate নিয়ে একটা lightweight cross-encoder দিয়ে পুনরায় স্কোর করে top-5 বাছাই করা হবে — যা Precision@5 (এখনো low, ~0.16-0.17) আরও বাড়ানোর জন্য।

ফলাফল দেখার পর জানাবেন — Stage 6-এ এগোব।


FINAL